In [1]:

import os 
import sys
from dataclasses import dataclass
sys.path.append(os.path.join(os.path.dirname("__file__"), '..'))

import numpy as np
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

import skfem
import gmsh
import meshio

from src.utils import *
from src.core import *

In [2]:
### 1-dim illusttrative case, with disconnected domain and blockage between two marginals (no solution)




In [9]:
def run_circle_obs_fig(config: Config, save_result=False, num_particle=30, n_snapshot=6):
    
    logger = Experiment(config.name_path)

    bound=(0, 2, 0, 1)
    circles= [((.9, .5), .4)]
    rounded_rects=[]
    nodes0, eles0 = build_dom_example(
        bound, circles, rounded_rects,
        h_min=config.h_min, h_max=config.h_max
    )

    dom = DOMobs1(3, bound, circles, rounded_rects)
    mesh = Mesh(nodes0, eles0, dom)
    E0 = Ekern(nodes0, gamma=config.gamma, sigma=config.sigma)

    def _make_marg(nodes, mass):
        x, y = nodes.T
        
        g1 = np.exp(-((x - 0.35) ** 2 + (y - 0.30) ** 2) / (2.0 * 0.08 ** 2))
        g2 = np.exp(-((x - 0.35) ** 2 + (y - 0.70) ** 2) / (2.0 * 0.08 ** 2))
        mu0= 0.5 * (g1 + g2) + 1e-12
        
        dx = np.maximum(0., np.maximum(1.5-x, x-1.8))
        dy = np.maximum(0., np.maximum(.35-y, y-.85))
        muT = np.exp(-(dx**2 + dy**2) / (2.*.05**2)) + 1e-12
        
        return mu0 / np.sum(mass*mu0), muT / np.sum(mass*muT)
    
    
    solution = solve_mfrsb(mesh, E0, SLStenc0, config, _make_marg, logger) 
    
    if save_result:
        box = [(1.45, 1.82, 0.20, 0.80)]
        trajs, max_penetration = path_sim(mesh, solution, num_particle, 3, config.seed)
        save_snapshot2d(mesh, solution, 
                        os.path.join(config.name_path, "snapshot.png"),
                        trajs, box, n_snapshot, "MFRSB with Circular Obs")
        logger.save_sol("solution_circle", mesh, solution, trajs, box)
        
    return solution, logger

In [12]:
config = Config(h_min=.028, h_max=.04, kappa=.07, gamma=.08)
output = run_circle_obs_fig(config, save_result=True)

  [Iter 01] ||drho||_inf: 5.40e+00 | q:    --- | L1(0,T): (5.61e-01, 5.24e-01) | W2: 4.31e-01 | MassErr: 4.4e-16 | MaxPen: 0.0e+00 | CFL: 0.03 | dt_CPU: 0.90s
  [Iter 02] ||drho||_inf: 9.35e+00 | q: 1.730 | L1(0,T): (1.96e-01, 1.83e-01) | W2: 2.33e-01 | MassErr: 5.6e-16 | MaxPen: 0.0e+00 | CFL: 0.08 | dt_CPU: 0.96s
  [Iter 03] ||drho||_inf: 3.55e+00 | q: 0.380 | L1(0,T): (6.87e-02, 6.42e-02) | W2: 1.20e-01 | MassErr: 6.7e-16 | MaxPen: 0.0e+00 | CFL: 0.09 | dt_CPU: 0.80s
  [Iter 04] ||drho||_inf: 1.39e+00 | q: 0.392 | L1(0,T): (2.40e-02, 2.25e-02) | W2: 5.37e-02 | MassErr: 6.7e-16 | MaxPen: 0.0e+00 | CFL: 0.10 | dt_CPU: 0.85s
  [Iter 05] ||drho||_inf: 5.05e-01 | q: 0.362 | L1(0,T): (8.42e-03, 7.86e-03) | W2: 3.88e-03 | MassErr: 4.4e-16 | MaxPen: 0.0e+00 | CFL: 0.10 | dt_CPU: 0.86s
  [Iter 06] ||drho||_inf: 1.82e-01 | q: 0.361 | L1(0,T): (2.95e-03, 2.75e-03) | W2: 2.30e-03 | MassErr: 6.7e-16 | MaxPen: 0.0e+00 | CFL: 0.10 | dt_CPU: 0.80s
  [Iter 07] ||drho||_inf: 6.75e-02 | q: 0.371 | L1(

In [ ]:

inc_ts = [1/10, 1/20, 1/40, 1/80]
inc_xs = [0.08, 0.04, 0.02, 0.01]


for inc_time in inc_ts:
    pass


for inc_x in inc_xs:
    pass


for inc_time, inc_x in zip(inc_ts, inc_xs):
    pass